<a href="https://colab.research.google.com/github/yosie111/rag_shul/blob/main/Shulchan_Aruch_Full_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Edited Table · Full Processing — from TXT to JSON

One notebook that includes both stages:
1. **Fix** — Cleaning tags, full spelling, unifying concepts, deleting references
2. **Stage 2 — JSON** — Building a JSON structure with text, hagah, text_raw

Upon completion, you receive a `shulchan_aruch.json` file ready for download.

## 1 · Loading the source file

In [1]:
import re, os, json
from collections import defaultdict, Counter

# ═══ Loading ═══
try:
    from google.colab import files as colab_files
    IN_COLAB = True
    print('🌐 Google Colab')
    uploaded = colab_files.upload()
    filename = list(uploaded.keys())[0]
except ImportError:
    IN_COLAB = False
    print('💻 Local execution')
    default_name = 'Shulchan_Arukh__Orach_Chayim_-_he_-_Torat_Emet_363.txt'
    filename = default_name if os.path.exists(default_name) else input('Path: ').strip()
    if not os.path.exists(filename):
        raise FileNotFoundError(filename)

def strip_nikud(s):
    return re.sub(r'[\u0591-\u05C7]', '', s)

with open(filename, 'r', encoding='utf-8') as f:
    lines = f.readlines()

n_lines_orig = len(lines)
n_siman = sum(1 for l in lines if re.match(r'^Siman \d+$', l.strip()))
print(f'✅ {filename}: {n_lines_orig:,} lines, {n_siman} simanim')

# ═══ Also keep original lines for Stage 2 (text_raw) ═══
original_lines = list(lines)

change_log = []


🌐 Google Colab


Saving Shulchan Arukh, Orach Chayim - he - Torat Emet 363.txt to Shulchan Arukh, Orach Chayim - he - Torat Emet 363.txt
✅ Shulchan Arukh, Orach Chayim - he - Torat Emet 363.txt: 10,205 lines, 697 simanim


## 2 · Basic syntax fixes
Orphaned tags, missing brackets, spaces, clause separation.

In [2]:
# ═══ 2.1 · Merging orphaned </small> tags ═══
fixed, orphan_count, i = [], 0, 0
while i < len(lines):
    if lines[i].strip() == '</small>':
        if fixed:
            fixed[-1] = fixed[-1].rstrip('\n') + '</small>\n'
            orphan_count += 1
        i += 1
    else:
        fixed.append(lines[i]); i += 1
lines = fixed

# ═══ 2.2 · Completing missing parentheses at depth 2 ═══
paren_count = 0
for i in range(len(lines)):
    parts = re.split(r'(<small>|</small>)', lines[i])
    depth = d2_o = d2_c = 0
    has_d2 = False
    for p in parts:
        if p == '<small>': depth += 1; continue
        if p == '</small>': depth -= 1; continue
        if depth >= 2 and p.strip():
            has_d2 = True
            d2_o += p.count('('); d2_c += p.count(')')
    if not has_d2 or d2_o <= d2_c:
        continue
    missing = d2_o - d2_c
    depth, last = 0, -1
    new_parts = []
    for p in parts:
        new_parts.append(p)
        if p == '<small>': depth += 1
        elif p == '</small>': depth -= 1
        elif depth >= 2 and p.strip():
            last = len(new_parts) - 1
    if last >= 0:
        new_parts[last] = new_parts[last].rstrip() + ')' * missing
        lines[i] = ''.join(new_parts)
        paren_count += 1

# ═══ 2.3 · Whitespace normalization ═══
ws_count = 0
for i in range(len(lines)):
    if not lines[i].strip():
        lines[i] = '\n'; continue
    new = re.sub(r' {2,}', ' ', lines[i]).rstrip() + '\n'
    if new != lines[i]:
        ws_count += 1; lines[i] = new

# ═══ 2.4 · Separating adjacent seifim ═══
result, prev_c, sep_count, started = [], False, 0, False
for line in lines:
    tr = line.strip()
    is_siman = bool(re.match(r'^Siman \d+$', tr))
    is_header = bool(re.match(r'^(Shulchan|שולחן|Torat|http)', tr))
    is_content = bool(tr) and not is_siman and not is_header
    if is_siman: started = True
    if started and is_content and prev_c:
        result.append('\n'); sep_count += 1
    result.append(line)
    if not tr or is_siman or is_header: prev_c = False
    elif is_content: prev_c = True
lines = result

balanced = sum(l.count('<small>') for l in lines) == sum(l.count('</small>') for l in lines)
print(f'✓ Orphaned tags:       {orphan_count}')
print(f'✓ Parens completed:    {paren_count}')
print(f'✓ Spaces cleaned:      {ws_count:,}')
print(f'✓ Seif separations:    {sep_count}')
print(f'✓ Tag balance:         {"OK" if balanced else "unbalanced"}')
change_log += [f'BASIC_FIX: orphans={orphan_count}, parens={paren_count}, ws={ws_count}, sep={sep_count}']


✓ Orphaned tags:       7
✓ Parens completed:    6
✓ Spaces cleaned:      4,136
✓ Seif separations:    220
✓ Tag balance:         OK


## 3 · Deep Cleaning 2+ — Removing `<small>`

In [3]:
# ═══ parse + render AST of <small> ═══

def parse_small_ast(line):
    """Parses a line into a tree of (type, content):
       ('text', str) | ('small', [children])
    """
    tokens = re.split(r'(<small>|</small>)', line)
    stack = [[]]
    for tok in tokens:
        if tok == '<small>':
            new = []
            stack[-1].append(('small', new))
            stack.append(new)
        elif tok == '</small>':
            if len(stack) > 1:
                stack.pop()
        else:
            stack[-1].append(('text', tok))
    return stack[0]


HEB_RE_LOCAL = re.compile(r'[\u05D0-\u05EA]')
EXPLANATION_RE = re.compile(r"^\(\s*(פי'|פירוש)\s*")

def is_citation(content):
    """Classifier for content at depth ≥ 2."""
    stripped = content.strip()
    if not stripped:
        return True
    # explanations — (פי' ...) — keep content, remove the word "פי'/פירוש"
    if EXPLANATION_RE.match(stripped):
        return False
    # short nesting fragments without Hebrew
    if len(stripped) <= 2 and not HEB_RE_LOCAL.search(stripped):
        return True
    # parentheses = reference
    if '(' in stripped or ')' in stripped:
        return True
    return False


def strip_pirush_word(text):
    """Removes the word פי'/פירוש from within parentheses, keeps the explanation content."""
    return re.sub(r"\(\s*(?:פי'|פירוש)\s*", '(', text)


def render_ast(ast, depth=1, audit=None, ctx=None):
    """Renders the tree back to string, deleting depth ≥ 2 classified as reference."""
    out = []
    for node_type, content in ast:
        if node_type == 'text':
            out.append(content)
            continue
        # small — recurse into children first
        inner = render_ast(content, depth + 1, audit, ctx)
        plain = re.sub(r'</?small>', '', inner)
        plain_norm = re.sub(r'\s+', ' ', plain).strip()
        if depth >= 2:
            drop = is_citation(plain_norm)
            if audit is not None:
                audit.append({
                    'action': 'DROP' if drop else 'KEEP',
                    'depth':  depth,
                    'siman':  ctx.get('siman') if ctx else None,
                    'line':   ctx.get('line_num') if ctx else None,
                    'content': plain_norm,
                })
            if drop:
                continue  # full removal of tags + content
        out.append('<small>' + inner + '</small>')
    return ''.join(out)


def strip_nested_citations(line, audit=None, ctx=None):
    ast = parse_small_ast(line)
    cleaned = render_ast(ast, depth=1, audit=audit, ctx=ctx)
    cleaned = re.sub(r'  +', ' ', cleaned)
    return cleaned


def strip_depth1_refs(line):
    """Removes <small> blocks at depth 1 that are mechaber references (not hagah).
    Uses the same state machine logic as parse_seif:
    - הגה → state = rema → keep
    - state == rema and no mechaber text → keep (continued Rema)
    - otherwise → mechaber reference → DROP
    """
    parts = re.split(r'(<small>|</small>)', line)
    depth = 0
    state = 'author'
    result = []
    ref_count = 0
    i = 0
    while i < len(parts):
        part = parts[i]
        if part == '<small>':
            depth += 1
            if depth == 1:
                # collect all block content until matching </small>
                inner_parts = [part]
                j = i + 1
                d = 1
                while j < len(parts) and d > 0:
                    inner_parts.append(parts[j])
                    if parts[j] == '<small>': d += 1
                    elif parts[j] == '</small>': d -= 1
                    j += 1
                # extract plain text (without tags and without nikud)
                plain = ''.join(p for p in inner_parts
                               if p not in ('<small>', '</small>'))
                plain = re.sub(r'[\u0591-\u05C7]', '', plain)
                plain = re.sub(r'\s+', ' ', plain).strip()
                if plain.startswith('הגה'):
                    state = 'rema'
                    result.extend(inner_parts)
                elif state == 'rema' and plain:
                    result.extend(inner_parts)
                else:
                    ref_count += 1  # mechaber reference → deleting
                i = j
                depth = 0
                continue
            else:
                result.append(part)
        elif part == '</small>':
            depth = max(0, depth - 1)
            result.append(part)
        else:
            if depth == 0 and part.strip() and HEB_RE_LOCAL.search(part):
                state = 'author'
            result.append(part)
        i += 1
    cleaned = ''.join(result)
    cleaned = re.sub(r'  +', ' ', cleaned)
    return cleaned, ref_count


# ═══ Running on all lines — depth 2+ and also depth 1 references ═══
audit_entries = []
current_siman = None
clean_count = 0
total_d1_refs = 0

for i, line in enumerate(lines):
    tr = line.strip()
    m = re.match(r'^Siman (\d+)$', tr)
    if m:
        current_siman = int(m.group(1))
        continue
    if not tr or re.match(r'^(Shulchan|שולחן|Torat|http)', tr):
        continue
    ctx = {'siman': current_siman, 'line_num': i + 1}
    # step 1: depth 2+ cleanup (sources)
    cleaned = strip_nested_citations(line, audit=audit_entries, ctx=ctx)
    # step 2: depth 1 cleanup (mechaber references)
    cleaned, d1_count = strip_depth1_refs(cleaned)
    total_d1_refs += d1_count
    if cleaned != line:
        clean_count += 1
        lines[i] = cleaned

# ═══ Statistics ═══
drops = [e for e in audit_entries if e['action'] == 'DROP']
keeps = [e for e in audit_entries if e['action'] == 'KEEP']

print(f'Lines cleaned:               {clean_count:,}')
print(f'<small> fragments at depth ≥ 2: {len(audit_entries):,}')
print(f'   Dropped (sources):        {len(drops):,}  ({100*len(drops)/max(len(audit_entries),1):.1f}%)')
print(f'   Kept (explanations):      {len(keeps):,}  ({100*len(keeps)/max(len(audit_entries),1):.1f}%)')
print(f'Mechaber refs at depth 1:    {total_d1_refs:,} deleted')

# detail of kept items — worth reviewing
if keeps:
    print(f'\n── All {len(keeps)} kept fragments (require review) ──')
    for e in keeps[:25]:
        tag = '(פי\')' if e['content'].startswith('(') else '(תוכן)'
        print(f'   [{e["depth"]}] {tag:<8} siman {e["siman"]}: {e["content"][:80]}')
    if len(keeps) > 25:
        print(f'   ... and {len(keeps) - 25} more')


change_log.append(f'SMALL_D2: {len(drops)} dropped, {len(keeps)} kept')


Lines cleaned:               1,879
<small> fragments at depth ≥ 2: 2,922
   Dropped (sources):        2,723  (93.2%)
   Kept (explanations):      199  (6.8%)
Mechaber refs at depth 1:    1,956 deleted

── All 199 kept fragments (require review) ──
   [2] (פי')    siman 3: (פי' מָקוֹם שֶׁיְּכוֹלִים מִשָּׁם לִרְאוֹת הַר הַבַּיִת וּמִשָּׁם וְהָלְאָה אֵין
   [2] (פי')    siman 11: (פי' הַחֵלֶק מֵהַצִיצִית שֶׁבֵּין קֶשֶׁר לְקֶשֶׁר)
   [2] (פי')    siman 17: (פי' אֵינוֹ חַיָּב לִקְנוֹת לוֹ טַלִּית כְּדֵי שֶׁיִּתְחַיֵּב בְּצִיצִית וּלְקַמ
   [2] (פי')    siman 17: (פי' טֻמְטוּם לֹא נוֹדַע אִם הוּא זָכָר אוֹ נְקֵבָה וְאַנְדְּרוֹגִינוּס יֵשׁ לוֹ
   [2] (פי')    siman 32: (פי' יְכַסֶה)
   [2] (פי')    siman 55: (פי' מְעַט כֹּתֶל יָשָׁר וְשָׁוֶה)
   [2] (פי')    siman 58: (פי' תַּלְמִידִים וְרַשִׁ"י פי' אֲנָשִׁים עֲנָוִים וּמְחַבְּבִים הַמִּצְוֹת)
   [2] (פי')    siman 58: (פי' יְצִיאַת הַחַמָּה כְּמוֹ הֵנֵצוּ הָרִמּוֹנִים)
   [2] (פי')    siman 97: (פי' שֶׁאֵינוֹ יָכוֹל לִרְאוֹת דְּבַר מִאוּס)
 

## 4 · Full spelling conversion (with "מים" exceptions)

In [4]:
# ═══ Nikud constants ═══
SHVA,   HATAF_S, HATAF_P, HATAF_Q = '\u05B0', '\u05B1', '\u05B2', '\u05B3'
HIRIQ,  TSERE,   SEGOL,   PATAH   = '\u05B4', '\u05B5', '\u05B6', '\u05B7'
QAMATS, HOLAM,   HOLAM_H, QUBUTS  = '\u05B8', '\u05B9', '\u05BA', '\u05BB'
DAGESH, SHIN_D,  SIN_D,   QAMATS_Q = '\u05BC', '\u05C1', '\u05C2', '\u05C7'

NIKUD_RE = re.compile(r'[\u05B0-\u05C7]')
HEB_RE   = re.compile(r'[\u05D0-\u05EA]')

# Short words that remain in ktiv haser (defective spelling)
SHORT_WORDS_NO_VAV = {
    'לא','הלא','ולא','שלא','בלא','דלא','אלא',
    'כל','וכל','בכל','לכל','מכל','שכל','דכל','ככל',
    'כך','וכך','לכך','על','של','אל',
    'עד','גם','כן','פן','אם','עם','מן','זה','זו','את',
}

# ⚠️ "מים" bug — exceptions to the dual suffix rule
DUAL_SUFFIX_EXCEPTIONS = {
    'מים', 'שמים', 'ירושלים', 'מצרים',
}

PE_ALEPH_FUTURE_PREFIXES = {'ת', 'י', 'נ'}


def tokenize(text):
    return re.findall(r'[\u05D0-\u05EA][\u05D0-\u05EA\u0591-\u05C7]*|[^\u05D0-\u05EA]+', text)


def parse_word(word):
    tokens, i = [], 0
    while i < len(word):
        ch = word[i]
        if HEB_RE.match(ch):
            marks, j = [], i + 1
            while j < len(word) and NIKUD_RE.match(word[j]):
                marks.append(word[j]); j += 1
            tokens.append((ch, set(marks))); i = j
        else:
            i += 1
    return tokens


def has_em_kriah_after(letters, idx):
    return idx + 1 < len(letters) and letters[idx + 1][0] in 'אעה'


def is_consonantal_vav(letters, idx):
    ch, marks = letters[idx]
    if ch != 'ו': return False
    return bool(marks & {QAMATS, PATAH, SHVA, HIRIQ, TSERE, SEGOL,
                          HATAF_P, HATAF_Q, HATAF_S})


def has_chirik_yod_mem_ending(letters, idx):
    if idx < 1 or idx + 1 >= len(letters): return False
    _, prev_marks = letters[idx - 1]
    cur_ch, cur_marks = letters[idx]
    next_ch, _ = letters[idx + 1]
    if cur_ch == 'י' and HIRIQ in cur_marks and next_ch in 'םמ':
        if PATAH in prev_marks or QAMATS in prev_marks:
            return True
    return False


def convert_word(word):
    plain = strip_nikud(word)
    # ktiv haser exceptions (לא/כל/של...)
    if plain in SHORT_WORDS_NO_VAV:
        return plain
    # ⚠️ dual suffix exceptions (מים/שמים/ירושלים/מצרים)
    if plain in DUAL_SUFFIX_EXCEPTIONS:
        return plain
    letters = parse_word(word)
    if not letters: return plain
    result = []
    n = len(letters)
    for idx, (ch, marks) in enumerate(letters):
        next_ch    = letters[idx + 1][0] if idx + 1 < n else None
        next_marks = letters[idx + 1][1] if idx + 1 < n else set()
        prev_ch    = letters[idx - 1][0] if idx > 0 else None
        result.append(ch)
        # consonantal vav mid-word → double vav
        if ch == 'ו' and 0 < idx < n - 1 and is_consonantal_vav(letters, idx):
            if prev_ch != 'ו' and next_ch != 'ו':
                result.append('ו')
        # kubutz → vav
        if QUBUTS in marks and ch != 'ו' and next_ch != 'ו':
            result.append('ו')
        # holam → vav
        if HOLAM in marks and ch != 'ו':
            add = True
            if has_em_kriah_after(letters, idx): add = False
            if (idx == 0 and ch in PE_ALEPH_FUTURE_PREFIXES
                and n >= 3 and letters[1][0] == 'א'): add = False
            if next_ch == 'ו': add = False
            if add: result.append('ו')
        # kamatz katan → vav
        if QAMATS_Q in marks and ch != 'ו' and next_ch != 'ו':
            result.append('ו')
        # kamatz + shva (kamatz katan in practice) → vav
        # ⚠️ dagesh on the letter itself = not kamatz katan (pi'el/nif'al/hitpa'el)
        if QAMATS in marks and SHVA in next_marks and DAGESH not in marks:
            if ch != 'ו' and next_ch != 'ו' and plain not in SHORT_WORDS_NO_VAV:
                if not (ch in 'בכלמשהוד'):
                    if not has_em_kriah_after(letters, idx):
                        result.append('ו')
        # dual suffix ַיִם → adding yod
        if has_chirik_yod_mem_ending(letters, idx):
            result.append('י')
        # ⚠️ consonantal yod mid-word → double yod (חייב, תייג)
        #    but not after prefixes (בַּיּוֹם → ביום)
        #    and not before vav (ציציּוֹת → ציציות, not ציצייות)
        if ch == 'י' and DAGESH in marks and 0 < idx < n - 1:
            if prev_ch != 'י' and next_ch != 'י' and next_ch != 'ו':
                all_prefix_before = all(letters[j][0] in 'בכלמשהוד' for j in range(idx))
                if not all_prefix_before:
                    result.append('י')
        # ⚠️ new: hirik before dagesh hazak → adding yod (תפילה, אפילו)
        #   idx > 0 prevents adding to prefixes (מִן, בִּן, לִ...)
        if idx > 0 and HIRIQ in marks and ch != 'י' and next_ch != 'י':
            if DAGESH in next_marks:
                result.append('י')
    return ''.join(result)


# ═══ Disambiguation table — semantic homographs (nikud → unambiguous form) ═══
DISAMBIG_TABLE = {
    # plain_form: [(nikud_condition_on_letters, replacement), ...]
    # first matching rule replaces. If no rule matches — normal conversion.
    'אחר': [
        # אַחַר (patah on chet) = אחרי (after)
        (lambda L: len(L) >= 2 and PATAH in L[1][1], 'אחרי'),
        # אַחֵר (tsere on resh) = אחר (other) — default, normal conversion
    ],
    'קדוש': [
        # קִדּוּשׁ (hirik on kuf) = קידוש (ritual)
        (lambda L: HIRIQ in L[0][1], 'קידוש'),
        # קָדוֹשׁ (kamatz) = קדוש (holy) — default
    ],
    'אסור': [
        # אִסּוּר (hirik on alef) = איסור (prohibition)
        (lambda L: HIRIQ in L[0][1], 'איסור'),
        # אָסוּר (kamatz) = אסור (forbidden) — default
    ],
    'עולה': [
        # עוֹלָה (kamatz on lamed) = עולה (burnt offering)
        (lambda L: len(L) >= 3 and QAMATS in L[2][1], 'עולה'),
        # עוֹלֶה (segol) = עולה (goes up) — same form in ktiv male
    ],
    'דין': [
        # דַּיָּן (patah+kamatz) = דיין (judge)
        (lambda L: len(L) >= 2 and PATAH in L[0][1] and (QAMATS in L[1][1] or PATAH in L[1][1]), 'דיין'),
        # דִּין (hirik) = דין (law) — default
    ],
    'מותר': [
        # מוֹתָר (kamatz on tav) = מותר (leftover/remainder)
        (lambda L: len(L) >= 3 and QAMATS in L[2][1] and HOLAM in L[0][1], 'מותר'),
        # מוּתָּר (shuruk) = מותר (permitted) — default
    ],
}


def nikud_to_ktiv_male(text):
    tokens = tokenize(text)
    result = []
    for tok in tokens:
        if not any(HEB_RE.match(c) for c in tok):
            result.append(strip_nikud(tok))
            continue
        plain = strip_nikud(tok)
        # check semantic disambiguation before normal conversion
        if plain in DISAMBIG_TABLE:
            letters = parse_word(tok)
            replaced = False
            for cond, replacement in DISAMBIG_TABLE[plain]:
                if letters and cond(letters):
                    result.append(replacement)
                    replaced = True
                    break
            if not replaced:
                result.append(convert_word(tok))
        else:
            result.append(convert_word(tok))
    return ''.join(result)


# ═══ Regression tests ═══
tests = [
    # v5 original
    ("יִתְגַּבֵּר כַּאֲרִי לַעֲמֹד בַּבֹּקֶר", "יתגבר כארי לעמוד בבוקר"),
    ("חָכְמָה",    "חוכמה"),
    ("קָרְבָּן",   "קורבן"),
    ("אָמְנָם",    "אומנם"),
    # v6 rules — consonantal vav + dual suffix
    ("מִצְוָה",    "מצווה"),
    ("תִּקְוָה",   "תקווה"),
    ("עֵינַיִם",   "עיניים"),
    ("רַגְלַיִם",  "רגליים"),
    ("יָדַיִם",    "ידיים"),
    ("אָזְנַיִם",  "אוזניים"),
    # ⚠️ new in v7 — "מים" exceptions
    ("מַיִם",          "מים"),
    ("שָׁמַיִם",       "שמים"),
    ("יְרוּשָׁלַיִם",  "ירושלים"),
    ("מִצְרַיִם",      "מצרים"),
    # gershayim preserved
    ('עַכּוּ"ם',   'עכו"ם'),
    ('רַמְבַּ"ם',  'רמב"ם'),
    ("ה'",         "ה'"),
    # ⚠️ fix prefix+kamatz+shva (not kamatz katan!)
    ("בָּרְכוּ",     "ברכו"),      # not בורכו
    ("הָרְצוּעוֹת",  "הרצועות"),   # not הורצועות
    # ⚠️ homograph disambiguation (nikud → unambiguous form)
    ("אַחַר",        "אחרי"),      # after (patah-patah)
    ("אַחֵר",        "אחר"),       # other (patah-tsere) — unchanged
    ("קִדּוּשׁ",     "קידוש"),     # ritual (hirik)
    ("קָדוֹשׁ",      "קדוש"),      # holy (kamatz) — unchanged
    ("אִסּוּר",      "איסור"),     # prohibition (hirik)
    ("אָסוּר",       "אסור"),      # forbidden (kamatz) — unchanged
    # two
    ("שְׁנַיִם",     "שניים"),     # two
    ("שָׁנִים",       "שנים"),      # years — unchanged
    # ⚠️ hirik → yod (תפילה, אפילו)
    ("תְּפִלָּה",     "תפילה"),
    ("אֲפִלּוּ",      "אפילו"),
    ("תְּפִלִּין",    "תפילין"),
    # ⚠️ consonantal yod → double yod (חייב)
    ("חַיָּב",         "חייב"),
    # ⚠️ dagesh+kamatz+shva = not kamatz katan (נתפרקו, גזרו)
    ("נִתְפָּרְקוּ",    "נתפרקו"),    # not נתפורקו
    ("גָּזְרוּ",        "גזרו"),      # not גוזרו
    # ⚠️ yod after prefix = not doubled (ביום, not בייום)
    ("בַּיּוֹם",         "ביום"),
    ("הַיּוֹם",          "היום"),
]
print('── Ktiv male converter test ──')
fails = 0
for menukad, expected in tests:
    got = nikud_to_ktiv_male(menukad)
    ok = got == expected
    fails += not ok
    mark = '✓' if ok else '✗'
    print(f'  {mark} {strip_nikud(menukad):>16} → {got:<16} (expected: {expected})')
print(f'\n  {len(tests)-fails}/{len(tests)} passing')
assert fails == 0, 'Converter tests failed!'


── Ktiv male converter test ──
  ✓ יתגבר כארי לעמד בבקר → יתגבר כארי לעמוד בבוקר (expected: יתגבר כארי לעמוד בבוקר)
  ✓             חכמה → חוכמה            (expected: חוכמה)
  ✓             קרבן → קורבן            (expected: קורבן)
  ✓             אמנם → אומנם            (expected: אומנם)
  ✓             מצוה → מצווה            (expected: מצווה)
  ✓             תקוה → תקווה            (expected: תקווה)
  ✓            עינים → עיניים           (expected: עיניים)
  ✓            רגלים → רגליים           (expected: רגליים)
  ✓             ידים → ידיים            (expected: ידיים)
  ✓            אזנים → אוזניים          (expected: אוזניים)
  ✓              מים → מים              (expected: מים)
  ✓             שמים → שמים             (expected: שמים)
  ✓          ירושלים → ירושלים          (expected: ירושלים)
  ✓            מצרים → מצרים            (expected: מצרים)
  ✓            עכו"ם → עכו"ם            (expected: עכו"ם)
  ✓            רמב"ם → רמב"ם            (expected: רמב"ם)
  ✓        

## 5 · Semantic unification — names of הקב"ה, גוי, deletion of references

In [5]:
# ═══ Service prefixes (letters of predication) ═══
# ב/כ/ל/מ/ש/ה/ו — can appear up to 3 in sequence (ולב-, ושל-, ומה-)
PREFIX_CLASS = r'[ובלמשכהד]'
PREFIX_UP_TO_3 = rf'{PREFIX_CLASS}{{0,3}}'
HEBREW = r'[\u05D0-\u05EA]'

def _word_bnd_pattern(term):
    """Regex with Hebrew word boundaries around a literal term."""
    return re.compile(rf'(?<!{HEBREW})' + re.escape(term) + rf'(?!{HEBREW})')

def _stem_with_prefix_pattern(stem):
    """Regex allowing service prefixes before the stem. Group 1 = the prefix."""
    return re.compile(rf'(?<!{HEBREW})({PREFIX_UP_TO_3}){re.escape(stem)}(?!{HEBREW})')


# ═══ Stem-based unification: prefix + STEM → prefix + TARGET ═══
# each stem gets uniform treatment across all possible prefixes
STEM_UNIFICATIONS = [
    # ── divine names ──
    ('הקב"ה',  'אלוהים'),
    ('השי"ת',  'אלוהים'),
    ('אלקים',  'אלוהים'),
    ('אלהים',  'אלוהים'),
    # ── non-Jewish ──
    ('עכו"ם',  'גוי'),
    ('כותי',   'גוי'),
    ('נכרי',   'גוי'),
    ('נכרים',  'גוים'),   # plural — preserving final mem
]

# ═══ Deletion (with prefixes) ═══
DELETE_STEMS = ["סי'", 'סעיף', 'עיין']
# Reference abbreviations deleted as whole words (no prefixes)
DELETE_WORDS = ['עי"ש', 'וע"ש']


# ═══ Context-aware ה' rule ═══
# Contexts where ה' is numeric/reference (not divine)
HEY_NOT_GOD_PREV = {
    'סעיף', 'סעיפים', 'פרק', 'פרקים', "פ'", 'פ"ק', 'סימן', 'סימנים',
    'כמנין', 'מנין', "סי'", 'הלכה', 'הלכות', 'ביום', 'יום',
    'באות', 'אות', 'דף', 'שער', 'שורה', 'עמוד', 'כלל',
}
HEY_NOT_GOD_NEXT = {
    'טפחים', 'אמות', 'מילין', 'פרסאות', 'שנים', 'שנה',
    'ימים', 'חדשים', 'שבועות', 'שעות', 'דקות',
    # with comma
    'טפחים,', 'אמות,', 'ימים,', 'שנים,',
}


def disambiguate_hey(text):
    """Replaces ה' → אלוהים only when context is *not* numeric/reference."""
    replacements = 0
    kept = 0
    def repl(m):
        nonlocal replacements, kept
        start, end = m.start(), m.end()
        # previous context word (up to 20 chars back)
        left = text[max(0, start - 20):start]
        prev_tokens = left.strip().split()
        prev_word = prev_tokens[-1] if prev_tokens else ''
        # next context word
        right = text[end:end + 20]
        next_tokens = right.strip().split()
        next_word = next_tokens[0] if next_tokens else ''
        # blacklist = not divine
        if prev_word in HEY_NOT_GOD_PREV or next_word in HEY_NOT_GOD_NEXT:
            kept += 1
            return m.group(0)
        replacements += 1
        return 'אלוהים'
    new_text = _word_bnd_pattern("ה'").sub(repl, text)
    return new_text, replacements, kept


def apply_stem_replacement(text, stem, target, stats_key_prefix):
    """prefix+stem → prefix+target. Returns (text, count)."""
    pat = _stem_with_prefix_pattern(stem)
    count = len(pat.findall(text))
    def repl(m):
        return m.group(1) + target
    return pat.sub(repl, text), count


def apply_stem_deletion(text, stem):
    """prefix+stem → "". Returns (text, count)."""
    pat = _stem_with_prefix_pattern(stem)
    count = len(pat.findall(text))
    return pat.sub('', text), count


def apply_synonym_unification(text):
    """Operation order:
       1. Context-aware ה' (requires סעיף/סי' to still exist)
       2. Stem unification (divine names, goy)
       3. Deleting full references (ע"ל/עיין + content) — before word deletion!
       4. Removing פי'/פירוש from parentheses — before word deletion!
       5. Stem deletion (סי'/סעיף/עיין)
       6. Whole word deletions (עי"ש/וע"ש)
    """
    stats = {}
    # ── 1: ה' ──
    text, hey_rep, hey_keep = disambiguate_hey(text)
    stats["HEY_→אלוהים"] = hey_rep
    stats["HEY_נשאר_מספרי"] = hey_keep
    # ── 2: stem unification ──
    for stem, tgt in STEM_UNIFICATIONS:
        text, count = apply_stem_replacement(text, stem, tgt, f'UNIFY_{stem}')
        if count:
            stats[f'UNIFY_{stem}→{tgt}'] = count
    # ── 3: full references (must run before individual word deletions!) ──
    # a. inside <small>: עיין/ע"ל/לקמן/לעיל + content → ""
    ref_in_small = re.compile(
        r'<small>\s*(?:ו?עיין|ו?ע"ל|ע"ל|ו?כד?לקמן|ו?כד?לעיל|ו?לקמן|ו?לעיל)\s*[^<]*?</small>'
    )
    c1 = len(ref_in_small.findall(text))
    text = ref_in_small.sub('', text)
    # b. outside <small>: עיין/ע"ל + content until period
    ref_bare_ayin = re.compile(
        r'(?<![\u05D0-\u05EA])(?:ו?עיין|ו?ע"ל)\s+[^.:\n<]{1,80}[.:]?'
    )
    c2 = len(ref_bare_ayin.findall(text))
    text = ref_bare_ayin.sub('', text)
    # c. כדלקמן/כדלעיל/לקמן/לעיל + reference content (siman/seif numbers)
    ref_lkm = re.compile(
        r'(?<![\u05D0-\u05EA])(?:ו?כד?לקמן|ו?כד?לעיל|ו?לקמן|ו?לעיל)\s+[^.,:;\n<]{0,80}[.,;:]?'
    )
    c3 = len(ref_lkm.findall(text))
    text = ref_lkm.sub('', text)
    # d. standalone כדלקמן/כדלעיל (no content after them)
    ref_standalone = re.compile(
        r'(?<![\u05D0-\u05EA])(?:ו?כדלקמן|ו?כדלעיל)(?![\u05D0-\u05EA])'
    )
    c4 = len(ref_standalone.findall(text))
    text = ref_standalone.sub('', text)
    stats['DELETE_הפניות'] = c1 + c2 + c3 + c4
    # ── 4: פי'/פירוש — exact operation order ──
    # 4a: expansion — פי' → פירוש (everywhere, including with prefixes: לפי' → לפירוש)
    expand_re = re.compile(r"פי'")
    expand_count = len(expand_re.findall(text))
    text = expand_re.sub('פירוש', text)
    # 4b: deletion — standalone פירוש only (not prefix+פירוש: לפירוש/כפירוש/בפירוש/הפירוש)
    delete_pirush = re.compile(r'(?<![\u05D0-\u05EA])פירוש\s*')
    delete_count = len(delete_pirush.findall(text))
    text = delete_pirush.sub('', text)
    stats["EXPAND_פי'→פירוש"] = expand_count
    stats["DELETE_פירוש"] = delete_count
    # ── 5: stems (סי'/סעיף/עיין) ──
    for stem in DELETE_STEMS:
        text, count = apply_stem_deletion(text, stem)
        stats[f'DELETE_{stem}'] = count
    # ── 6: whole words (עי"ש/וע"ש) ──
    for word in DELETE_WORDS:
        pat = _word_bnd_pattern(word)
        count = len(pat.findall(text))
        text = pat.sub('', text)
        if count:
            stats[f'DELETE_{word}'] = count
    # ── 7: expanding numeric abbreviations by context ──
    MEASURE_WORDS_RE = r'(?:אמות|טפחים|כוסות|מינים|כנפות|פרסאות|מילין|ציציות|ברכות|פעמים|שעות|ימים|חדשים|שבועות|אצבעות|זיתים|ביצים|גריסין|רביעיות)'
    for letter, number in [("ד'", 'ארבע'), ("ג'", 'שלוש'), ("ב'", 'שתי')]:
        pat = re.compile(re.escape(letter) + r'\s+(' + MEASURE_WORDS_RE + ')')
        count = len(pat.findall(text))
        text = pat.sub(number + r' \1', text)
        if count:
            stats[f'EXPAND_{letter}→{number}'] = count
    # ── 8: references to Shulchan Aruch sections → deletion ──
    SECTION_REFS = ['יו"ד', 'חו"מ', 'או"ח', 'אה"ע', 'יורה דעה', 'חושן משפט', 'אורח חיים', 'אבן העזר']
    for ref in SECTION_REFS:
        pat = _word_bnd_pattern(ref)
        c = len(pat.findall(text))
        text = pat.sub('', text)
        if c: stats[f'DELETE_{ref}'] = c
    # ── 9: expanding known abbreviations ──
    for abbr, expansion in [('וה"ה', 'והוא הדין'), ('ה"ה', 'הוא הדין'),
                             ('נ"ל', 'הנזכר לעיל'),
                             ('ג"ט', 'שלושה טפחים'), ('ד"ט', 'ארבעה טפחים')]:
        pat = _word_bnd_pattern(abbr)
        c = len(pat.findall(text))
        text = pat.sub(expansion, text)
        if c: stats[f'EXPAND_{abbr}'] = c
    # ── 10: converting numeric abbreviations → number words ──
    GEMATRIA = {
        'א':1,'ב':2,'ג':3,'ד':4,'ה':5,'ו':6,'ז':7,'ח':8,'ט':9,
        'י':10,'כ':20,'ל':30,'מ':40,'נ':50,'ס':60,'ע':70,'פ':80,'צ':90,
        'ק':100,'ר':200,'ש':300,'ת':400,
    }
    UNITS_F = ['','אחת','שתיים','שלוש','ארבע','חמש','שש','שבע','שמונה','תשע']
    UNITS_M = ['','אחד','שניים','שלושה','ארבעה','חמישה','שישה','שבעה','שמונה','תשעה']
    TENS    = ['','עשר','עשרים','שלושים','ארבעים','חמישים','ששים','שבעים','שמונים','תשעים']
    TEENS_F = ['עשר','אחת עשרה','שתים עשרה','שלוש עשרה','ארבע עשרה','חמש עשרה',
               'שש עשרה','שבע עשרה','שמונה עשרה','תשע עשרה']
    TEENS_M = ['עשרה','אחד עשר','שנים עשר','שלושה עשר','ארבעה עשר','חמישה עשר',
               'שישה עשר','שבעה עשר','שמונה עשר','תשעה עשר']
    HUNDREDS = ['','מאה','מאתיים','שלוש מאות','ארבע מאות','חמש מאות']
    def gematria_val(letters):
        return sum(GEMATRIA.get(c, 0) for c in letters)
    def num_to_heb(n, fem=False):
        if n <= 0 or n > 599: return None
        u = UNITS_F if fem else UNITS_M
        t_teens = TEENS_F if fem else TEENS_M
        parts = []
        if n >= 100:
            h = n // 100
            if h < len(HUNDREDS): parts.append(HUNDREDS[h])
            n %= 100
        if 10 <= n <= 19:
            parts.append(t_teens[n - 10])
        else:
            if n >= 20:
                parts.append(TENS[n // 10])
            if n % 10 > 0:
                parts.append(u[n % 10])
        return ' '.join(parts) if parts else None
    # (number expansion moved to step 0 in pipeline — where original nikud is accessible)
    # ── whitespace cleanup ──
    text = re.sub(r'  +', ' ', text)
    text = re.sub(r'^ +| +$', '', text, flags=re.MULTILINE)
    return text, stats


# ═══ Regression tests ═══
tests_unify = [
    # divine name unification
    ('שהרי הקב"ה מקדים לברך',      'שהרי אלוהים מקדים לברך'),
    ('הבורא יתברך השי"ת ברא',       'הבורא יתברך אלוהים ברא'),
    ('להקב"ה אין גבול',             'לאלוהים אין גבול'),
    # context-aware ה'
    ("חייב לברך את ה' אלהינו",      'חייב לברך את אלוהים אלהינו'),
    ("סעיף ה' מסכם",                "ה' מסכם"),
    ("ה' טפחים",                    "ה' טפחים"),
    ("ברוך ה'",                     'ברוך אלוהים'),
    # goy with prefixes
    ('עכו"ם שבא לבית',               'גוי שבא לבית'),
    ('העכו"ם מחמתו',                 'הגוי מחמתו'),
    ('ולעכו"ם לא נותנים',            'ולגוי לא נותנים'),
    ('כותי אחד ונכרי אחר',           'גוי אחד וגוי אחר'),
    # deleting full references
    ('<small> וע"ל מ"ז י"ג. </small>',                 ''),
    ('טקסט לפני <small> ע"ל סימן ל"ג. </small> אחרי', 'טקסט לפני אחרי'),
    ('טקסט ועיין בסימן מ"ז.',                          'טקסט'),
    # references כדלקמן/כדלעיל/לקמן/לעיל
    ('כדלקמן סימן רט"ו',                               ''),
    ('ברכות שמברך קודם נטילה כדלעיל סימן מ"ו.',        'ברכות שמברך קודם נטילה'),
    ('ביקר לעיל סימן י"א ב\'. טקסט.',                 'ביקר טקסט.'),
    ('אמר לקמן סימן כ"ה י"ב.',                          'אמר'),
    # removing פי' — inside parentheses and outside
    ("(פי' מקום שיכולים לראות)",      "(מקום שיכולים לראות)"),
    ("(פירוש תלמידים)",              "(תלמידים)"),
    ("רבנו תם פי' שהוא כעין עטרה",   "רבנו תם שהוא כעין עטרה"),
    # לפי'/לפירוש — preposition, do not delete
    ("דברי עצמו לפי' הטור",          "דברי עצמו לפירוש הטור"),
    # expanding numeric abbreviations
    ("ד' אמות",                      "ארבע אמות"),
    ("ג' פעמים",                     "שלוש פעמים"),
    ("ב' כוסות",                     "שתי כוסות"),
    ("ד' על ד' טפחים",              "ד' על ארבע טפחים"),
]
print('── Unification tests ──')
fails = 0
for inp, expected in tests_unify:
    out, _ = apply_synonym_unification(inp)
    ok = out.strip() == expected.strip()
    fails += not ok
    mark = '✓' if ok else '✗'
    print(f'  {mark} {inp!r}')
    print(f'     → {out!r}')
    if not ok:
        print(f'     ✗ expected: {expected!r}')
print(f'\n  {len(tests_unify)-fails}/{len(tests_unify)} passing')
if fails: raise AssertionError(f'{fails} tests failed')


── Unification tests ──
  ✓ 'שהרי הקב"ה מקדים לברך'
     → 'שהרי אלוהים מקדים לברך'
  ✓ 'הבורא יתברך השי"ת ברא'
     → 'הבורא יתברך אלוהים ברא'
  ✓ 'להקב"ה אין גבול'
     → 'לאלוהים אין גבול'
  ✓ "חייב לברך את ה' אלהינו"
     → 'חייב לברך את אלוהים אלהינו'
  ✓ "סעיף ה' מסכם"
     → "ה' מסכם"
  ✓ "ה' טפחים"
     → "ה' טפחים"
  ✓ "ברוך ה'"
     → 'ברוך אלוהים'
  ✓ 'עכו"ם שבא לבית'
     → 'גוי שבא לבית'
  ✓ 'העכו"ם מחמתו'
     → 'הגוי מחמתו'
  ✓ 'ולעכו"ם לא נותנים'
     → 'ולגוי לא נותנים'
  ✓ 'כותי אחד ונכרי אחר'
     → 'גוי אחד וגוי אחר'
  ✓ '<small> וע"ל מ"ז י"ג. </small>'
     → ''
  ✓ 'טקסט לפני <small> ע"ל סימן ל"ג. </small> אחרי'
     → 'טקסט לפני אחרי'
  ✓ 'טקסט ועיין בסימן מ"ז.'
     → 'טקסט'
  ✓ 'כדלקמן סימן רט"ו'
     → ''
  ✓ 'ברכות שמברך קודם נטילה כדלעיל סימן מ"ו.'
     → 'ברכות שמברך קודם נטילה'
  ✓ 'ביקר לעיל סימן י"א ב\'. טקסט.'
     → 'ביקר טקסט.'
  ✓ 'אמר לקמן סימן כ"ה י"ב.'
     → 'אמר'
  ✓ "(פי' מקום שיכולים לראות)"
     → '(מקום שיכולים לראות)'
  ✓ '(פירוש תלמידים)'


## 6 · Pipeline — Processing all rows

In [6]:
# ═══ Pipeline ═══
processed_lines = []
content_lines_count = 0
total_unification_stats = Counter()

for line in lines:
    tr = line.strip()
    # headers and blank lines — no change
    if not tr or re.match(r'^(Siman \d+|Shulchan|שולחן|Torat|http)', tr):
        processed_lines.append(line)
        continue
    # content line — processing
    content_lines_count += 1
    # ── step 0: expanding numeric abbreviations on the vowelized text ──
    # golden rule: if letters have nikud = sage name, no nikud = number
    NIKUD_CHECK = re.compile(r'[\u05B0-\u05C7]')
    NUM_ABBREV_NIKUD = re.compile(
        r'[\u05D0-\u05EA\u0591-\u05C7]{1,6}"[\u05D0-\u05EA\u0591-\u05C7]{1,3}'
    )
    GEMATRIA_V = {
        'א':1,'ב':2,'ג':3,'ד':4,'ה':5,'ו':6,'ז':7,'ח':8,'ט':9,
        'י':10,'כ':20,'ל':30,'מ':40,'נ':50,'ס':60,'ע':70,'פ':80,'צ':90,
        'ק':100,'ר':200,'ש':300,'ת':400,
    }
    U_M = ['','אחד','שניים','שלושה','ארבעה','חמישה','שישה','שבעה','שמונה','תשעה']
    T_M = ['עשרה','אחד עשר','שנים עשר','שלושה עשר','ארבעה עשר','חמישה עשר',
           'שישה עשר','שבעה עשר','שמונה עשר','תשעה עשר']
    TENS_W = ['','עשר','עשרים','שלושים','ארבעים','חמישים','ששים','שבעים','שמונים','תשעים']
    def _gval(s): return sum(GEMATRIA_V.get(c,0) for c in s if 'א' <= c <= 'ת')
    def _n2h(n):
        if n < 2 or n > 999: return None
        if n >= 100:
            h = n // 100
            HUND = ['','מאה','מאתיים','שלוש מאות','ארבע מאות',
                    'חמש מאות','שש מאות','שבע מאות','שמונה מאות','תשע מאות']
            if h >= len(HUND): return None
            rest = n % 100
            if rest == 0: return HUND[h]
            rest_h = _n2h(rest)
            return f'{HUND[h]} {rest_h}' if rest_h else None
        if 10 <= n <= 19: return T_M[n-10]
        parts = []
        if n >= 20: parts.append(TENS_W[n//10])
        if n % 10: parts.append(U_M[n%10])
        return ' '.join(parts)
    HUNDREDS_SET = set('קרשת')
    TENS_SET     = set('יכלמנסעפצ')
    UNITS_SET    = set('אבגדהוזחט')
    def _valid_gematria(chars):
        """Checks valid gematria structure: strict descending order."""
        if not chars or len(chars) > 5: return False
        vals = [GEMATRIA_V.get(c,0) for c in chars]
        if any(v == 0 for v in vals): return False
        for i in range(len(vals)-1):
            if vals[i] <= vals[i+1]: return False
        return True
    NOT_NUMBERS = {'נ"ל','ה"ה','וה"ה','ד"מ','ר"ת','ש"ע','ש"ץ','נ"ך','ס"ת','ע"ב','ח"ל'}
    NIKUD_MARKS = set(chr(c) for c in range(0x05B0, 0x05C8))
    QAMATS_C = '\u05B8'
    num_expand_count = [0]
    def _expand_num_in_nikud(m):
        tok = m.group(0)
        plain = strip_nikud(tok)
        if plain in NOT_NUMBERS:
            return tok
        # parsing letters: (char, set of nikud)
        parsed = []
        i = 0
        while i < len(tok):
            if '\u05D0' <= tok[i] <= '\u05EA':
                marks = set(); j = i + 1
                while j < len(tok) and tok[j] in NIKUD_MARKS:
                    marks.add(tok[j]); j += 1
                parsed.append((tok[i], marks)); i = j
            else:
                i += 1
        # separating prefixes (vowelized) from core (unvowelized)
        prefix_end = 0
        for idx, (ch, marks) in enumerate(parsed):
            if marks and ch in 'בכלמשהוד':
                prefix_end = idx + 1
            else:
                break
        # ⚠️ prefix vav with kamatz = noun (וָי"ו = letter name), don't convert
        # prefix vav with shva = conjunction vav (וְי"ד = ו+14), do convert
        for idx in range(prefix_end):
            ch, marks = parsed[idx]
            if ch == 'ו' and QAMATS_C in marks:
                return tok  # kamatz on vav = noun → not a number
        core = parsed[prefix_end:]
        # all core letters must be without nikud
        if any(marks for _, marks in core):
            return tok
        core_chars = [ch for ch, _ in core]
        # checking valid gematria structure
        if not _valid_gematria(core_chars):
            return tok
        val = _gval(''.join(core_chars))
        heb = _n2h(val)
        if heb and 2 <= val <= 99:
            num_expand_count[0] += 1
            # keep vowelized prefixes (without nikud)
            prefix_str = ''.join(ch for ch, _ in parsed[:prefix_end])
            return prefix_str + heb
        return tok
    line = NUM_ABBREV_NIKUD.sub(_expand_num_in_nikud, line)
    total_unification_stats['EXPAND_מספרים'] += num_expand_count[0]
    # ── step 1: ktiv male (preserves <small>) ──
    converted = nikud_to_ktiv_male(line)
    # step 2: semantic unification
    unified, stats = apply_synonym_unification(converted)
    for k, v in stats.items():
        total_unification_stats[k] += v
    # ensure trailing newline
    if not unified.endswith('\n'):
        unified += '\n'
    processed_lines.append(unified)

# ═══ Final cleanup — whitespace and redundant punctuation ═══
clean_stats = Counter()
for i, line in enumerate(processed_lines):
    orig = line
    # 1. removing <small> with punctuation only — no Hebrew (source remnants)
    line = re.sub(r'<small>\s*([^<]{0,5})\s*</small>',
                  lambda m: '' if not re.search(r'[\u05D0-\u05EA]', m.group(1)) else m.group(0),
                  line)
    # 2. removing empty <small></small>
    line = re.sub(r'<small>\s*</small>', '', line)
    # 3. space before punctuation
    line = re.sub(r' +([.,:;])', r'\1', line)
    # 4. double punctuation (.. → .)
    line = re.sub(r'([.,:;])\1+', r'\1', line)
    # 5. double spaces
    line = re.sub(r'  +', ' ', line)
    # 6. leading/trailing space (preserves newline)
    line = line.strip() + '\n' if line.strip() else '\n'
    if line != orig:
        clean_stats['שורות_נוקו'] += 1
    processed_lines[i] = line

# 7. multiple blank lines → single blank line
final_lines = []
prev_blank = False
for line in processed_lines:
    is_blank = not line.strip()
    if is_blank and prev_blank:
        continue
    final_lines.append(line)
    prev_blank = is_blank
processed_lines = final_lines

print(f'── Final cleanup ──')
print(f'  Lines cleaned:      {clean_stats["שורות_נוקו"]:,}')
print(f'  Lines before:       {len(processed_lines) + (len(processed_lines) - len(final_lines)):,}')
print(f'  Lines after:        {len(processed_lines):,}')


print(f'── Pipeline complete ──')
print(f'  Content lines processed: {content_lines_count:,}')
print(f'  Output lines:            {len(processed_lines):,}')

# ═══ Unification report ═══
print('\n── Unification report ──')
for k, v in sorted(total_unification_stats.items(), key=lambda x: -x[1]):
    if v > 0:
        print(f'  {v:>5}  {k}')


── Final cleanup ──
  Lines cleaned:      1,111
  Lines before:       9,736
  Lines after:        9,736
── Pipeline complete ──
  Content lines processed: 4,168
  Output lines:            9,736

── Unification report ──
    400  UNIFY_עכו"ם→גוי
    280  EXPAND_מספרים
    226  DELETE_הפניות
    140  HEY_→אלוהים
    129  EXPAND_ד'→ארבע
     67  EXPAND_ג'→שלוש
     25  EXPAND_ב'→שתי
     20  DELETE_סי'
     15  HEY_נשאר_מספרי
     15  UNIFY_הקב"ה→אלוהים
     15  UNIFY_כותי→גוי
     14  EXPAND_וה"ה
     10  EXPAND_נ"ל
      9  UNIFY_אלהים→אלוהים
      9  EXPAND_ג"ט
      8  DELETE_סעיף
      7  EXPAND_פי'→פירוש
      7  DELETE_פירוש
      7  DELETE_יו"ד
      6  EXPAND_ד"ט
      3  DELETE_עיין
      2  UNIFY_נכרים→גוים
      2  DELETE_יורה דעה
      1  UNIFY_נכרי→גוי


## 7 · Step 2 — Building JSON
Compress text_raw (מנוקד), text (processed), hagah (הגה) for each section.

In [7]:
# ═══ Parsing functions for Stage 2 ═══

def merge_orphan_tags_s2(lines):
    fixed = []
    for line in lines:
        if line.strip() == '</small>' and fixed:
            fixed[-1] = fixed[-1].rstrip('\n') + '</small>\n'
        else:
            fixed.append(line)
    return fixed


def parse_torat_emet_to_seifim(lines_list, fix_orphans=False):
    if fix_orphans:
        lines_list = merge_orphan_tags_s2(lines_list)
    result = []
    current_siman = None
    current_seifim = []
    for line in lines_list:
        tr = line.strip()
        m = re.match(r'^Siman (\d+)$', tr)
        if m:
            if current_siman is not None:
                result.append((current_siman, current_seifim))
            current_siman = int(m.group(1))
            current_seifim = []
            continue
        if not tr or re.match(r'^(Shulchan|שולחן|Torat|http)', tr):
            continue
        if not re.sub(r'</?small>', '', tr).strip():
            continue
        current_seifim.append(line.rstrip('\n'))
    if current_siman is not None:
        result.append((current_siman, current_seifim))
    return result

HEB_RE = re.compile(r'[\u05D0-\u05EA]')

def parse_seif(normalized_line):
    """Parses a normalized line into {text, hagah}."""
    parts = re.split(r'(<small>|</small>)', normalized_line)
    depth = 0
    mechaber_parts, hagah_parts = [], []
    state = 'author'  # 'author' | 'rema'

    for part in parts:
        if part == '<small>':
            depth += 1; continue
        if part == '</small>':
            depth = max(0, depth - 1); continue
        if depth == 0:
            mechaber_parts.append(part)
            if part.strip() and HEB_RE.search(part):
                state = 'author'
        elif depth == 1:
            stripped = part.strip()
            if stripped.startswith('הגה'):
                state = 'rema'
                hagah_parts.append(part)
            elif state == 'rema':
                hagah_parts.append(part)
            # depth 1 that is not hagah = mechaber reference → skip (not in template)
        # depth ≥ 2 — already cleaned in stage 1, should not appear

    def clean(ps):
        return re.sub(r'\s+', ' ', ''.join(ps)).strip()

    text = clean(mechaber_parts).strip(': ')
    hagah = clean(hagah_parts)
    # removes *all* "הגה:" — including internal ones (template-compliant)
    hagah = re.sub(r'(?<![\u05D0-\u05EA])הגה\s*:\s*', '', hagah).strip()

    return {
        'text':  text  or None,
        'hagah': hagah or None,
    }


# ═══ Test ═══
sample = 'טקסט מחבר <small> הגה: דברי רמ"א חלק א </small> <small> הגה: דברי רמ"א חלק ב </small>'
p = parse_seif(sample)
print('── parse_seif test ──')
print(f'  text:  {p["text"]}')
print(f'  hagah: {p["hagah"]}')
assert p['hagah'] == 'דברי רמ"א חלק א דברי רמ"א חלק ב', f'Failed: {p["hagah"]}'
print('  ✓ Two hagah blocks merged without internal "הגה:"')

# ═══ Build data from original and processed lines ═══
raw_data  = parse_torat_emet_to_seifim(original_lines, fix_orphans=True)
norm_data = parse_torat_emet_to_seifim(processed_lines, fix_orphans=False)

print(f'── Alignment check ──')
print(f'  Simanim in source:     {len(raw_data)}')
print(f'  Simanim in normalized: {len(norm_data)}')
print(f'  Seifim in source:      {sum(len(s) for _, s in raw_data):,}')
print(f'  Seifim in normalized:  {sum(len(s) for _, s in norm_data):,}')

assert len(raw_data) == len(norm_data), \
    f'Siman count mismatch: source {len(raw_data)}, normalized {len(norm_data)}'

misaligned = []
for (s_r, sf_r), (s_n, sf_n) in zip(raw_data, norm_data):
    if s_r != s_n:
        misaligned.append(f'Different siman number: {s_r} vs {s_n}')
    if len(sf_r) != len(sf_n):
        misaligned.append(f'Siman {s_r}: {len(sf_r)} seifim in source, {len(sf_n)} in normalized')

if misaligned:
    print(f'\n❌ {len(misaligned)} mismatches:')
    for m in misaligned[:10]: print(f'   {m}')
    raise AssertionError('Source-normalized alignment failed')
else:
    print('  ✓ Perfect internal alignment')


── parse_seif test ──
  text:  טקסט מחבר
  hagah: דברי רמ"א חלק א דברי רמ"א חלק ב
  ✓ Two hagah blocks merged without internal "הגה:"
── Alignment check ──
  Simanim in source:     697
  Simanim in normalized: 697
  Seifim in source:      4,168
  Seifim in normalized:  4,168
  ✓ Perfect internal alignment


## 8 · Generating the JSON and downloading

In [8]:
simanim = []

for (siman_num, raw_lines), (_, norm_lines) in zip(raw_data, norm_data):
    seifim = []
    for i, (raw, norm) in enumerate(zip(raw_lines, norm_lines)):
        parsed = parse_seif(norm)
        seifim.append({
            'seif':     i + 1,
            'text':     parsed['text'],
            'hagah':    parsed['hagah'],
            'text_raw': raw,   # unprocessed source — nikud + all tags
        })
    simanim.append({'siman': siman_num, 'seifim': seifim})

output = {
    'title':   'שולחן ערוך, אורח חיים',
    'source':  'Torat Emet 363',
    'simanim': simanim,
}

# ═══ Summary ═══
total_seifim = sum(len(s['seifim']) for s in simanim)
hagah_count = sum(1 for s in simanim for sf in s['seifim'] if sf['hagah'])
empty_text  = sum(1 for s in simanim for sf in s['seifim'] if not sf['text'])

print(f'Simanim:           {len(simanim)}')
print(f'Seifim:            {total_seifim:,}')
print(f'   with hagah:     {hagah_count:,}  ({100*hagah_count/total_seifim:.1f}%)')
print(f'   without text:   {empty_text}')

# ═══ Save ═══
out_path = 'shulchan_aruch.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

size_mb = os.path.getsize(out_path) / (1024*1024)
print(f'\n📁 {out_path} ({size_mb:.2f} MB)')


print('═' * 72)
print(' ' * 16 + 'JSON display — matches shulchan_aruch_template.json')
print('═' * 72)

for so in simanim[:2]:
    print(f'\n▍ siman {so["siman"]}  ({len(so["seifim"])} seifim)')
    print('─' * 72)
    for sf in so['seifim'][:2]:
        print(f'   seif {sf["seif"]}:')
        print(f'      text:  {(sf["text"] or "")[:90]}...')
        if sf['hagah']:
            print(f'      hagah: {sf["hagah"][:90]}...')
        else:
            print(f'      hagah: null')
        raw_preview = sf['text_raw'][:120].replace('\n', ' ')
        print(f'      text_raw: {raw_preview}...')
        print()

print('─' * 72)
print(f'Output file: shulchan_aruch.json')
print(f'Load directly with: `json.load(open("shulchan_aruch.json"))`')
print('─' * 72)


# ═══ Download in Colab ═══
try:
    from google.colab import files as colab_files
    colab_files.download(out_path)
    print('⬇️ Downloading shulchan_aruch.json...')
except ImportError:
    print(f'\n✅ Done — output file: {out_path}')


Simanim:           697
Seifim:            4,168
   with hagah:     1,304  (31.3%)
   without text:   0

📁 shulchan_aruch.json (4.83 MB)
════════════════════════════════════════════════════════════════════════
                JSON display — matches shulchan_aruch_template.json
════════════════════════════════════════════════════════════════════════

▍ siman 1  (9 seifim)
────────────────────────────────────────────────────────────────────────
   seif 1:
      text:  יתגבר כארי לעמוד בבוקר לעבודת בוראו שיהא הוא מעורר השחר...
      hagah: ועל כל פנים לא יאחר זמן התפילה שהציבור מתפללין. שוויתי אלוהים לנגדי תמיד הוא כלל גדול בתור...
      text_raw: יִתְגַּבֵּר כַּאֲרִי לַעֲמֹד בַּבֹּקֶר לַעֲבוֹדַת בּוֹרְאוֹ שֶׁיְּהֵא הוּא מְעוֹרֵר הַשַּׁחַר: <small> הַגָּה: וְעַל כָּ...

   seif 2:
      text:  המשכים להתחנן לפני בוראו, יכוון לשעות שמשתנות המשמרות שהן בשליש הלילה ולסוף שני שלישי הליל...
      hagah: null
      text_raw: הַמַּשְׁכִּים לְהִתְחַנֵּן לִפְנֵי בּוֹרְאוֹ, יְכַוֵּן לַשָּׁעוֹת שֶׁמּ

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ Downloading shulchan_aruch.json...
